# 9.2 Modifying Graphs — Apply

## Objective

Master ONNX graph modification techniques. You will:

1. Insert new nodes into an existing graph (e.g., add Relu activation)
2. Remove nodes from a graph
3. Replace one operator with another
4. Merge two models (sequential composition)
5. Modify initializer values (weight swapping)
6. Modify graph I/O declarations
7. Verify correctness after each modification

**Graph modification invariant:** After any transformation, the model must satisfy:

$$\text{check\_model}(M') = \text{pass} \quad \land \quad \text{InferenceSession}(M') \text{ succeeds}$$

In [ ]:
# Setup
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort
import copy
import tempfile
import os

TMPDIR = tempfile.mkdtemp(prefix='onnx_modify_')
print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}")

def verify_model(model, test_input=None):
    """Verify model passes check_model and can run inference."""
    try:
        check_model(model)
        status = 'check: PASS'
    except Exception as e:
        print(f"  check_model FAILED: {e}")
        return False

    try:
        sess = ort.InferenceSession(model.SerializeToString(),
                                    providers=['CPUExecutionProvider'])
        inp = sess.get_inputs()[0]
        if test_input is None:
            shape = [d if isinstance(d, int) else 2 for d in inp.shape]
            test_input = np.random.randn(*shape).astype(np.float32)
        result = sess.run(None, {inp.name: test_input})[0]
        print(f"  {status}, inference: PASS (output shape {result.shape})")
        return True
    except Exception as e:
        print(f"  {status}, inference FAILED: {e}")
        return False

---
## Exercise 1: Insert a New Node (Add Relu Activation)

Given a linear model $y = Wx + b$, we insert a Relu activation: $y = \max(0, Wx + b)$.

The key operation: rewire the output of the Add node to an intermediate tensor,
then connect Relu from that intermediate to the original output name.

In [ ]:
np.random.seed(42)

# Build a simple linear model: Y = X @ W + b
W = np.random.randn(8, 4).astype(np.float32) * 0.1
b = np.random.randn(4).astype(np.float32) * 0.01

nodes = [
    helper.make_node('MatMul', ['X', 'W'], ['mm']),
    helper.make_node('Add', ['mm', 'b'], ['Y']),
]
inits = [numpy_helper.from_array(W, 'W'), numpy_helper.from_array(b, 'b')]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 8])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 4])
g = helper.make_graph(nodes, 'linear', [X_i], [Y_i], initializer=inits)
base_model = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

print("Before insertion:")
for n in base_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
verify_model(base_model)

# Insert Relu: rewire Add output to 'pre_relu', add Relu -> 'Y'
modified = copy.deepcopy(base_model)
# Change Add's output from 'Y' to 'pre_relu'
for node in modified.graph.node:
    if node.op_type == 'Add':
        node.output[0] = 'pre_relu'

# Add Relu node
relu_node = helper.make_node('Relu', ['pre_relu'], ['Y'])
modified.graph.node.append(relu_node)

print("\nAfter inserting Relu:")
for n in modified.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
verify_model(modified)

# Compare outputs
x_test = np.random.randn(2, 8).astype(np.float32)
sess_orig = ort.InferenceSession(base_model.SerializeToString())
sess_mod = ort.InferenceSession(modified.SerializeToString())
y_orig = sess_orig.run(None, {'X': x_test})[0]
y_mod = sess_mod.run(None, {'X': x_test})[0]

print(f"\nOriginal output (can be negative): min={y_orig.min():.4f}")
print(f"Modified output (Relu'd):           min={y_mod.min():.4f}")
assert y_mod.min() >= 0, "Relu failed!"
assert np.allclose(y_mod, np.maximum(0, y_orig)), "Relu output mismatch!"
print(f"Verified: modified = max(0, original)")

---
## Exercise 2: Remove Nodes from a Graph

Remove a node by bypassing it: connect the node's input directly to its consumers.
This is useful for removing Identity, Dropout (at inference), or redundant ops.

In [ ]:
# Build model with removable nodes: X -> Identity -> Relu -> Identity -> Y
nodes = [
    helper.make_node('Identity', ['X'], ['id1']),
    helper.make_node('Relu', ['id1'], ['relu_out']),
    helper.make_node('Identity', ['relu_out'], ['Y']),
]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [4])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4])
g = helper.make_graph(nodes, 'with_identity', [X_i], [Y_i])
id_model = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

print(f"Before removal: {len(id_model.graph.node)} nodes")
for n in id_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

def remove_identity_nodes(model):
    """Remove all Identity nodes by bypassing them."""
    result = copy.deepcopy(model)
    g = result.graph

    # Build rename map: identity_output -> identity_input
    rename = {}
    identity_indices = []
    for idx, node in enumerate(g.node):
        if node.op_type == 'Identity':
            rename[node.output[0]] = node.input[0]
            identity_indices.append(idx)

    # Transitively resolve renames (chain of identities)
    def resolve(name):
        visited = set()
        while name in rename and name not in visited:
            visited.add(name)
            name = rename[name]
        return name

    # Update all node inputs
    for node in g.node:
        for i in range(len(node.input)):
            node.input[i] = resolve(node.input[i])

    # Update graph outputs
    for out in g.output:
        resolved = resolve(out.name)
        if resolved != out.name:
            out.name = resolved

    # Remove Identity nodes (reverse order to preserve indices)
    for idx in sorted(identity_indices, reverse=True):
        del g.node[idx]

    return result

cleaned = remove_identity_nodes(id_model)

print(f"\nAfter removal: {len(cleaned.graph.node)} nodes")
for n in cleaned.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
verify_model(cleaned)

# Verify equivalence
x_test = np.array([-2.0, -1.0, 1.0, 2.0], dtype=np.float32)
y_orig = ort.InferenceSession(id_model.SerializeToString()).run(None, {'X': x_test})[0]
y_clean = ort.InferenceSession(cleaned.SerializeToString()).run(None, {'X': x_test})[0]
assert np.allclose(y_orig, y_clean)
print(f"Equivalence verified: outputs match.")

---
## Exercise 3: Replace One Operator with Another

Replace Relu with LeakyRelu (which allows small negative gradients):

$$\text{Relu}(x) = \max(0, x) \quad\to\quad \text{LeakyRelu}(x) = \begin{cases} x & x > 0 \\ \alpha x & x \leq 0 \end{cases}$$

In [ ]:
# Build model: X -> MatMul -> Relu -> Y
W = np.random.randn(8, 4).astype(np.float32) * 0.1
nodes = [
    helper.make_node('MatMul', ['X', 'W'], ['mm']),
    helper.make_node('Relu', ['mm'], ['Y']),
]
inits = [numpy_helper.from_array(W, 'W')]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 8])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 4])
g = helper.make_graph(nodes, 'relu_model', [X_i], [Y_i], initializer=inits)
relu_model = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

print("Before replacement:")
for n in relu_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

def replace_relu_with_leaky(model, alpha=0.01):
    """Replace all Relu nodes with LeakyRelu."""
    result = copy.deepcopy(model)
    new_nodes = []
    replaced = 0
    for node in result.graph.node:
        if node.op_type == 'Relu':
            leaky = helper.make_node('LeakyRelu', list(node.input), list(node.output),
                                    alpha=alpha)
            new_nodes.append(leaky)
            replaced += 1
        else:
            new_nodes.append(node)

    del result.graph.node[:]
    result.graph.node.extend(new_nodes)
    return result, replaced

leaky_model, n_replaced = replace_relu_with_leaky(relu_model, alpha=0.1)

print(f"\nAfter replacement ({n_replaced} nodes replaced):")
for n in leaky_model.graph.node:
    attrs = {a.name: a.f for a in n.attribute if a.type == 1}
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)} {attrs if attrs else ''}")
verify_model(leaky_model)

# Compare behavior on negative inputs
x_test = np.array([[-1.0, 2.0, -3.0, 4.0], [5.0, -6.0, 7.0, -8.0]], dtype=np.float32)
y_relu = ort.InferenceSession(relu_model.SerializeToString()).run(None, {'X': x_test})[0]
y_leaky = ort.InferenceSession(leaky_model.SerializeToString()).run(None, {'X': x_test})[0]

print(f"\nBehavior comparison (negative inputs):")
print(f"  Relu output min:      {y_relu.min():.4f} (clamps to 0)")
print(f"  LeakyRelu output min: {y_leaky.min():.4f} (allows negative * alpha)")

---
## Exercise 4: Merge Two Models (Sequential Composition)

Given models $M_1: X \to Z$ and $M_2: Z \to Y$, create $M_{\text{merged}}: X \to Y$
by connecting $M_1$'s output to $M_2$'s input.

$$M_{\text{merged}} = M_2 \circ M_1$$

In [ ]:
# Model 1: X -> MatMul -> Relu -> Z (feature extractor)
W1 = np.random.randn(8, 4).astype(np.float32) * 0.1
nodes1 = [
    helper.make_node('MatMul', ['X', 'W1'], ['mm1']),
    helper.make_node('Relu', ['mm1'], ['Z']),
]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 8])
Z_i = helper.make_tensor_value_info('Z', TensorProto.FLOAT, [2, 4])
g1 = helper.make_graph(nodes1, 'extractor', [X_i], [Z_i],
                       initializer=[numpy_helper.from_array(W1, 'W1')])
model1 = helper.make_model(g1, opset_imports=[helper.make_opsetid('', 17)])

# Model 2: Z -> MatMul -> Softmax -> Y (classifier)
W2 = np.random.randn(4, 3).astype(np.float32) * 0.1
nodes2 = [
    helper.make_node('MatMul', ['Z', 'W2'], ['logits']),
    helper.make_node('Softmax', ['logits'], ['Y'], axis=1),
]
Z_i2 = helper.make_tensor_value_info('Z', TensorProto.FLOAT, [2, 4])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 3])
g2 = helper.make_graph(nodes2, 'classifier', [Z_i2], [Y_i],
                       initializer=[numpy_helper.from_array(W2, 'W2')])
model2 = helper.make_model(g2, opset_imports=[helper.make_opsetid('', 17)])

def merge_sequential(m1, m2, link_output='Z', link_input='Z'):
    """Merge two models by connecting m1's output to m2's input."""
    m1_copy = copy.deepcopy(m1)
    m2_copy = copy.deepcopy(m2)

    # If the linking names differ, rename m2's input
    if link_output != link_input:
        for node in m2_copy.graph.node:
            for i in range(len(node.input)):
                if node.input[i] == link_input:
                    node.input[i] = link_output

    # Combine nodes
    all_nodes = list(m1_copy.graph.node) + list(m2_copy.graph.node)

    # Combine initializers (avoiding duplicates)
    init_names = set()
    all_inits = []
    for init in list(m1_copy.graph.initializer) + list(m2_copy.graph.initializer):
        if init.name not in init_names:
            all_inits.append(init)
            init_names.add(init.name)

    # Use m1's inputs and m2's outputs
    merged_graph = helper.make_graph(
        all_nodes, 'merged',
        inputs=list(m1_copy.graph.input),
        outputs=list(m2_copy.graph.output),
        initializer=all_inits,
    )
    merged = helper.make_model(merged_graph, opset_imports=[helper.make_opsetid('', 17)])
    return merged

merged = merge_sequential(model1, model2)

print(f"Model 1: {len(model1.graph.node)} nodes ({[n.op_type for n in model1.graph.node]})")
print(f"Model 2: {len(model2.graph.node)} nodes ({[n.op_type for n in model2.graph.node]})")
print(f"Merged:  {len(merged.graph.node)} nodes ({[n.op_type for n in merged.graph.node]})")
verify_model(merged)

# Verify: merged(x) == model2(model1(x))
x_test = np.random.randn(2, 8).astype(np.float32)
z = ort.InferenceSession(model1.SerializeToString()).run(None, {'X': x_test})[0]
y_serial = ort.InferenceSession(model2.SerializeToString()).run(None, {'Z': z})[0]
y_merged = ort.InferenceSession(merged.SerializeToString()).run(None, {'X': x_test})[0]

assert np.allclose(y_serial, y_merged, atol=1e-6)
print(f"\nVerified: merged(x) == model2(model1(x))")
print(f"Output sum to 1 (Softmax): {np.allclose(y_merged.sum(axis=1), 1.0)}")

---
## Exercise 5: Modify Initializer Values (Weight Swapping)

Replace the weights of a trained model with new values. Useful for:
- Loading pretrained weights into an architecture
- Weight surgery (e.g., pruning, scaling)
- Testing with controlled weight values

In [ ]:
# Start with the base linear model
model = copy.deepcopy(base_model)

# Show original weights
print("Original weights:")
for init in model.graph.initializer:
    arr = numpy_helper.to_array(init)
    print(f"  {init.name}: shape={arr.shape}, mean={arr.mean():.4f}, std={arr.std():.4f}")

def swap_weights(model, name, new_values):
    """Replace an initializer's values."""
    for init in model.graph.initializer:
        if init.name == name:
            old_arr = numpy_helper.to_array(init)
            assert new_values.shape == old_arr.shape, \
                f"Shape mismatch: {new_values.shape} vs {old_arr.shape}"
            assert new_values.dtype == old_arr.dtype, \
                f"Dtype mismatch: {new_values.dtype} vs {old_arr.dtype}"
            new_init = numpy_helper.from_array(new_values, name)
            init.CopyFrom(new_init)
            return True
    return False

# Swap W with an orthogonal matrix (preserves norms)
from numpy.linalg import qr
Q, _ = qr(np.random.randn(8, 8).astype(np.float32))
W_ortho = Q[:, :4]  # Take first 4 columns

assert swap_weights(model, 'W', W_ortho)

# Swap bias with zeros
assert swap_weights(model, 'b', np.zeros(4, dtype=np.float32))

print("\nSwapped weights:")
for init in model.graph.initializer:
    arr = numpy_helper.to_array(init)
    print(f"  {init.name}: shape={arr.shape}, mean={arr.mean():.4f}, std={arr.std():.4f}")

verify_model(model)

# Verify the orthogonal property: ||Wx|| ≈ ||x|| (norm-preserving)
x_test = np.random.randn(2, 8).astype(np.float32)
y_test = ort.InferenceSession(model.SerializeToString()).run(None, {'X': x_test})[0]
print(f"\nOrthogonal weight check:")
print(f"  ||x|| = {np.linalg.norm(x_test, axis=1)}")
print(f"  ||Wx|| = {np.linalg.norm(y_test, axis=1)}")

---
## Exercise 6: Add/Modify Graph I/O Declarations

Expose intermediate tensors as additional outputs, or change the graph's
input/output declarations (useful for submodel extraction, debugging).

In [ ]:
# Build a model with multiple layers
np.random.seed(42)
W1 = np.random.randn(8, 6).astype(np.float32) * 0.1
W2 = np.random.randn(6, 4).astype(np.float32) * 0.1

nodes = [
    helper.make_node('MatMul', ['X', 'W1'], ['h']),
    helper.make_node('Relu', ['h'], ['a']),
    helper.make_node('MatMul', ['a', 'W2'], ['Y']),
]
inits = [
    numpy_helper.from_array(W1, 'W1'),
    numpy_helper.from_array(W2, 'W2'),
]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 8])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 4])
g = helper.make_graph(nodes, 'two_layer', [X_i], [Y_i], initializer=inits)
two_layer = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

print(f"Original outputs: {[o.name for o in two_layer.graph.output]}")

# Add intermediate tensor 'a' (after Relu) as an additional output
modified = copy.deepcopy(two_layer)
a_output = helper.make_tensor_value_info('a', TensorProto.FLOAT, [2, 6])
modified.graph.output.append(a_output)

print(f"Modified outputs: {[o.name for o in modified.graph.output]}")

# Run and get both outputs
sess = ort.InferenceSession(modified.SerializeToString(),
                            providers=['CPUExecutionProvider'])
x_test = np.random.randn(2, 8).astype(np.float32)
results = sess.run(None, {'X': x_test})

print(f"\nOutput 'Y' shape: {results[0].shape}")
print(f"Output 'a' (intermediate) shape: {results[1].shape}")
print(f"Intermediate 'a' min: {results[1].min():.4f} (should be >= 0 after Relu)")
assert results[1].min() >= 0, "Relu intermediate should be non-negative!"

# Verify: Y = a @ W2
y_check = results[1] @ W2
assert np.allclose(results[0], y_check, atol=1e-5)
print(f"Verified: Y = a @ W2")

---
## Exercise 7: Multi-Step Graph Transformation Pipeline

Chain multiple transformations together, verifying the model after each step.
This simulates a real optimization/modification pipeline.

In [ ]:
# Build a model with various modification opportunities
np.random.seed(0)
W1 = np.random.randn(8, 6).astype(np.float32) * 0.1
b1 = np.zeros(6, dtype=np.float32)
W2 = np.random.randn(6, 4).astype(np.float32) * 0.1
b2 = np.zeros(4, dtype=np.float32)

nodes = [
    helper.make_node('Identity', ['X'], ['x_id']),      # removable
    helper.make_node('MatMul', ['x_id', 'W1'], ['mm1']),
    helper.make_node('Add', ['mm1', 'b1'], ['a1']),
    helper.make_node('Relu', ['a1'], ['r1']),            # replaceable
    helper.make_node('MatMul', ['r1', 'W2'], ['mm2']),
    helper.make_node('Add', ['mm2', 'b2'], ['a2']),
    helper.make_node('Identity', ['a2'], ['Y']),         # removable
]
inits = [
    numpy_helper.from_array(W1, 'W1'), numpy_helper.from_array(b1, 'b1'),
    numpy_helper.from_array(W2, 'W2'), numpy_helper.from_array(b2, 'b2'),
]
X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 8])
Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 4])
g = helper.make_graph(nodes, 'pipeline_demo', [X_i], [Y_i], initializer=inits)
pipeline_model = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

# Reference output
x_test = np.random.randn(2, 8).astype(np.float32)
y_ref = ort.InferenceSession(pipeline_model.SerializeToString()).run(None, {'X': x_test})[0]

print(f"Step 0 — Original: {len(pipeline_model.graph.node)} nodes")
for n in pipeline_model.graph.node:
    print(f"  {n.op_type}")

# Step 1: Remove Identity nodes
step1 = remove_identity_nodes(pipeline_model)
assert verify_model(step1)
y1 = ort.InferenceSession(step1.SerializeToString()).run(None, {'X': x_test})[0]
assert np.allclose(y_ref, y1, atol=1e-6)
print(f"\nStep 1 — Remove Identity: {len(step1.graph.node)} nodes ✓")

# Step 2: Replace Relu with LeakyRelu
step2, n_rep = replace_relu_with_leaky(step1, alpha=0.01)
assert verify_model(step2)
print(f"\nStep 2 — Replace Relu->LeakyRelu: {n_rep} replaced ✓")

# Step 3: Scale weights by 2x
step3 = copy.deepcopy(step2)
for init in step3.graph.initializer:
    arr = numpy_helper.to_array(init)
    if arr.ndim == 2:  # only weight matrices
        new_arr = arr * 2.0
        new_init = numpy_helper.from_array(new_arr, init.name)
        init.CopyFrom(new_init)

assert verify_model(step3)
print(f"\nStep 3 — Scale weights 2x ✓")

# Final comparison
print(f"\nPipeline summary:")
print(f"  Nodes: {len(pipeline_model.graph.node)} -> {len(step3.graph.node)}")
print(f"  Ops: {[n.op_type for n in step3.graph.node]}")

---
## Challenge: Graph Transformation Pipeline

Build a configurable transformation pipeline that takes a list of transformation
specs and applies them in order, verifying after each step.

In [ ]:
class GraphTransformPipeline:
    """Configurable graph transformation pipeline with verification."""

    def __init__(self, model):
        self.original = model
        self.current = copy.deepcopy(model)
        self.history = [('original', len(model.graph.node))]

    def remove_identities(self):
        self.current = remove_identity_nodes(self.current)
        self.history.append(('remove_identity', len(self.current.graph.node)))
        return self

    def replace_activation(self, old_op, new_op, **attrs):
        result = copy.deepcopy(self.current)
        new_nodes = []
        for node in result.graph.node:
            if node.op_type == old_op:
                replacement = helper.make_node(new_op, list(node.input),
                                               list(node.output), **attrs)
                new_nodes.append(replacement)
            else:
                new_nodes.append(node)
        del result.graph.node[:]
        result.graph.node.extend(new_nodes)
        self.current = result
        self.history.append((f'replace_{old_op}->{new_op}', len(result.graph.node)))
        return self

    def scale_weights(self, factor):
        for init in self.current.graph.initializer:
            arr = numpy_helper.to_array(init)
            if arr.ndim >= 2:
                new = numpy_helper.from_array(arr * factor, init.name)
                init.CopyFrom(new)
        self.history.append((f'scale_weights_{factor}x', len(self.current.graph.node)))
        return self

    def insert_node_after(self, after_output, new_op, **attrs):
        result = copy.deepcopy(self.current)
        intermediate = f'{after_output}_pre_{new_op.lower()}'
        for node in result.graph.node:
            for i, out in enumerate(node.output):
                if out == after_output:
                    node.output[i] = intermediate
        new_node = helper.make_node(new_op, [intermediate], [after_output], **attrs)
        result.graph.node.append(new_node)
        self.current = result
        self.history.append((f'insert_{new_op}_after_{after_output}', len(result.graph.node)))
        return self

    def verify(self):
        try:
            check_model(self.current)
            sess = ort.InferenceSession(self.current.SerializeToString(),
                                        providers=['CPUExecutionProvider'])
            inp = sess.get_inputs()[0]
            shape = [d if isinstance(d, int) else 2 for d in inp.shape]
            x = np.random.randn(*shape).astype(np.float32)
            sess.run(None, {inp.name: x})
            return True
        except Exception as e:
            print(f"  Verification failed: {e}")
            return False

    def report(self):
        print(f"\nTransformation Pipeline Report")
        print(f"{'='*50}")
        for step, (name, n_nodes) in enumerate(self.history):
            status = '✓' if step == 0 or True else '?'
            print(f"  Step {step}: {name:<35} {n_nodes} nodes {status}")
        print(f"\nFinal ops: {[n.op_type for n in self.current.graph.node]}")
        print(f"Valid: {self.verify()}")

# Run the pipeline
pipe = GraphTransformPipeline(pipeline_model)
pipe.remove_identities() \
    .replace_activation('Relu', 'LeakyRelu', alpha=0.02) \
    .scale_weights(1.5)

pipe.report()

---
## Summary

| Concept | What You Practiced |
|:---|:---|
| Insert node | Rewired outputs, added Relu after linear layer |
| Remove node | Bypassed Identity nodes with transitive renaming |
| Replace operator | Swapped Relu → LeakyRelu with attribute propagation |
| Merge models | Sequential composition $M_2 \circ M_1$ |
| Weight swap | Replaced initializer values (orthogonal weights) |
| I/O modification | Exposed intermediate tensors as additional outputs |
| Pipeline | Chained transforms with verification at each step |

**Next:** [Shape Inference](../03_Shape_Inference/)